# Unconditional SEDD sampling + generative perplexity

Generate unconditional text with SEDD and score it under GPT-2 Large, the same
*generative perplexity* metric the training loop uses (`run_train.py`).

This compares the vanilla analytic sampler against the **`lb_mean_field`** corrector
(parallel locally-balanced Metropolis-Hastings), including a **matched-NFE** comparison.

**Requirements:** a CUDA GPU and `flash_attn` (the SEDD transformer needs it).
The default checkpoint `louaaron/sedd-medium` is downloaded from the HF Hub on first run.

## 1. Setup

In [ ]:
import torch
import torch.nn.functional as F
from transformers import GPT2TokenizerFast, GPT2LMHeadModel

import sampling
from load_model import load_model

# ---- config ----
MODEL_PATH = 'louaaron/sedd-medium'   # or a local run dir
SEQ_LEN    = 1024                      # SEDD / GPT-2 context length
BATCH_SIZE = 8                         # sequences to generate per run
SEED       = 42

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

In [ ]:
model, graph, noise = load_model(MODEL_PATH, device)
model.eval()
tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')

# External evaluator for generative perplexity (matches run_train.py).
eval_model = GPT2LMHeadModel.from_pretrained('gpt2-large').to(device).eval()
print('loaded SEDD + GPT-2 Large evaluator')

## 2. Helpers

`generate` wraps `sampling.get_pc_sampler`; `corrector_steps=0` (default) reproduces the
vanilla sampler exactly. `generative_perplexity` is copied from the eval block in
`run_train.py`: per-position cross-entropy under GPT-2 Large, mean over tokens, `exp`, mean over the batch.

In [ ]:
@torch.no_grad()
def generate(steps, corrector='none', corrector_steps=0, balancing='barker',
             update_fraction=1.0, corrector_t_threshold=0.0, batch_size=BATCH_SIZE,
             seed=SEED):
    if seed is not None:
        torch.manual_seed(seed)
    sampling_fn = sampling.get_pc_sampler(
        graph, noise, (batch_size, SEQ_LEN), 'analytic', steps, device=device,
        corrector=corrector, corrector_steps=corrector_steps, balancing=balancing,
        update_fraction=update_fraction, corrector_t_threshold=corrector_t_threshold,
    )
    return sampling_fn(model)   # LongTensor [batch_size, SEQ_LEN] of token ids


@torch.no_grad()
def generative_perplexity(samples, ppl_batch_size=4):
    """Generative perplexity under GPT-2 Large (lower is better)."""
    batches = max(1, samples.shape[0] // ppl_batch_size)
    total = 0.0
    for i in range(batches):
        s = samples[i * ppl_batch_size:(i + 1) * ppl_batch_size]
        _, logits = eval_model(s, labels=s)[:2]
        logits = logits.transpose(-1, -2)
        ppl = F.cross_entropy(logits[..., :-1], s[..., 1:], reduction='none').mean(dim=-1).exp().mean()
        total += ppl.item()
    return total / batches


def nfe(steps, corrector_steps=0):
    """Score-function calls: analytic predictor = 1/step, corrector = 2/step."""
    return steps * (1 + 2 * corrector_steps)


def show(samples, n=2):
    for txt in tokenizer.batch_decode(samples[:n]):
        print(txt)
        print('=' * 80)

## 3. Vanilla SEDD baseline

In [ ]:
STEPS = 128

vanilla = generate(steps=STEPS, corrector='none')
vanilla_ppl = generative_perplexity(vanilla)
print(f'vanilla   | steps={STEPS} NFE={nfe(STEPS)} | gen-ppl = {vanilla_ppl:.3f}')
show(vanilla)

## 4. `lb_mean_field` corrector

Adds `corrector_steps` parallel locally-balanced MH steps per noise level (2 score calls each).
Try `balancing='sqrt'`, `update_fraction<1.0`, or a `corrector_t_threshold` to gate the corrector
to higher noise levels.

In [ ]:
K = 1
corrected = generate(steps=STEPS, corrector='lb_mean_field', corrector_steps=K,
                     balancing='barker', update_fraction=1.0)
corrected_ppl = generative_perplexity(corrected)
print(f'lb K={K}    | steps={STEPS} NFE={nfe(STEPS, K)} | gen-ppl = {corrected_ppl:.3f}')
show(corrected)

## 5. Matched-NFE comparison

The corrector costs `1 + 2K` score calls per level, so for a fair comparison hold total NFE
fixed: vanilla with `N` levels vs corrector with `~N/3` levels and `K=1` (both `~N` calls).
The question is whether refinement at fewer levels beats pure Euler at more levels.

In [ ]:
BUDGET = 384   # target NFE budget

configs = [
    dict(label='vanilla',      steps=BUDGET,        corrector='none',          corrector_steps=0),
    dict(label='lb K=1',       steps=BUDGET // 3,   corrector='lb_mean_field', corrector_steps=1),
    dict(label='lb K=1 sqrt',  steps=BUDGET // 3,   corrector='lb_mean_field', corrector_steps=1, balancing='sqrt'),
    dict(label='lb K=1 uf0.5', steps=BUDGET // 3,   corrector='lb_mean_field', corrector_steps=1, update_fraction=0.5),
]

print(f'{"config":<14}{"steps":>7}{"NFE":>7}{"gen-ppl":>10}')
print('-' * 38)
for c in configs:
    label = c.pop('label')
    s = generate(**c)
    ppl = generative_perplexity(s)
    print(f'{label:<14}{c["steps"]:>7}{nfe(c["steps"], c["corrector_steps"]):>7}{ppl:>10.3f}')

## Notes

- **Lower generative perplexity is not unambiguously better** — extremely low perplexity can indicate
  degenerate/repetitive text. Inspect the decoded samples alongside the numbers.
- The corrector targets SEDD's mean-field factorization, so near `t=0` (low noise) parallel updates can
  produce locally inconsistent tokens. Mitigate with `corrector_t_threshold` (e.g. `0.2`) or
  `update_fraction < 1.0`.
- `corrector_steps=0` is bit-identical to the vanilla sampler.